# Day 2: Data Quality & Cleaning Pipeline
**Hakeem • SparkCity • Local analysis**

Deliverables: a before/after quality assessment, reusable cleaning functions, IQR and Z-score outlier detection/treatment, and validated Parquet datasets for Day 3.

Run all cells in order using the project's `.venv` kernel after `uv sync --dev`. Input is a completed Day 1 Parquet snapshot in `data/processed/day1`. Run the updated Day 1 notebook through its final export cells first. Each run creates a new directory in `data/processed/day2`; raw files and S2 are never changed.

The pipeline uses `sparkcityx.data_quality` as the shared contract. Passing that contract is evidence of structural quality, not proof that measurements are accurate or local data equals S2. Statistical extremes may represent real events.

## 1. Environment and run configuration
The root is discovered from your working directory, whether Jupyter starts in the repository root or `notebooks`. Java must be available; the setup recognizes this Mac's Homebrew Java 17 installation. Timestamps have no documented source timezone: retain their wall-clock values, without claiming UTC conversion.

In [9]:
import os
import sys
import json
import math
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import NumericType, StringType
from sparkcityx.loaders import load_dataset
from sparkcityx.data_quality import get_validation_config, validate_dataframe

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/sparkcityx").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside SparkCity_Capstone.")
java_home = Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home")
if not os.getenv("JAVA_HOME") and java_home.exists():
    os.environ["JAVA_HOME"] = str(java_home)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
spark = (SparkSession.builder.master("local[2]").appName("Hakeem-Day2")
         .config("spark.ui.enabled", "false")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.sql.session.timeZone", "UTC").getOrCreate())
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sparkContext.setLogLevel("ERROR")

def input_hash(path):
    """Hash file bytes and relative names, including all files in Parquet directories."""
    digest = hashlib.sha256()
    paths = sorted(path.rglob("*")) if path.is_dir() else [path]
    for item in paths:
        if item.is_file():
            if path.is_dir():
                digest.update(str(item.relative_to(path)).encode())
            with item.open("rb") as handle:
                for block in iter(lambda: handle.read(1024 * 1024), b""):
                    digest.update(block)
    return digest.hexdigest()

DAY1_RUN = None  # Set to a specific Day 1 directory name to pin the input snapshot.
base = ROOT / "data/processed/day1"
complete = sorted(p.parent for p in base.glob("*/manifest.json")
                  if json.loads(p.read_text()).get("status") == "complete")
DAY1 = base / DAY1_RUN if DAY1_RUN else (complete[-1] if complete else None)
if DAY1 is None or not (DAY1 / "manifest.json").exists():
    raise RuntimeError("Run the updated Hakeem_day1.ipynb through its final export cells first.")
source_manifest = json.loads((DAY1 / "manifest.json").read_text())
if source_manifest.get("status") != "complete" or source_manifest.get("stage") != "day1":
    raise ValueError("The selected snapshot is not a completed Day 1 export.")
source_records = {r["dataset"]: r for r in source_manifest["datasets"]}
expected_kinds = {"traffic", "air_quality", "weather", "energy", "city_zones", "occupancy", "fiscal"}
if set(source_records) != expected_kinds:
    raise ValueError("Day 1 must supply exactly the seven supported datasets.")
INPUT_PATHS = {kind: DAY1 / record["path"] for kind, record in source_records.items()}
for kind, path in INPUT_PATHS.items():
    if not path.is_dir() or input_hash(path) != source_records[kind]["sha256"]:
        raise ValueError(f"{kind}: Day 1 snapshot missing or modified; rerun Day 1.")
source_manifest_hash = hashlib.sha256((DAY1 / "manifest.json").read_bytes()).hexdigest()
print("Day 1 source snapshot:", DAY1)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = ROOT / "data/processed/day2" / RUN_ID
# None means unknown: only set these after confirming the data contract.
SOURCE_UNITS = dict(source_manifest["source_units"])
# Safe default: retain statistical extremes with flags.
OUTLIER_TREATMENT = "flag"  # "clip" is available only for approved columns below.
CLIP_COLUMNS = {"traffic": ["avg_speed"], "air_quality": ["pm25", "pm10", "no2", "co"],
                "weather": ["wind_speed", "pressure"], "energy": ["power_consumption"]}
print("Python:", sys.executable)
print("Spark:", spark.version)
print("Output:", OUTPUT)

Day 1 source snapshot: /Users/hakeem/Projects/SparkCity_Capstone/data/processed/day1/20260914T135810735977Z
Python: /Users/hakeem/Projects/SparkCity_Capstone/.venv/bin/python
Spark: 4.2.0
Output: /Users/hakeem/Projects/SparkCity_Capstone/data/processed/day2/20260914T135902169587Z


## 2. Day 1 input quality assessment
Report required fields, nulls, key duplicates, numeric summaries, non-finite values, blank strings, and shared domain rules. Temporal intervals are **observed per sensor**, not a promised reporting schedule. The generator rotates sensor IDs, so a five-minute dataset step does not imply five-minute reporting for each sensor.

This reports interval distributions; declaring an outage or interpolating entire missing rows requires an agreed reporting schedule. Whole-dataset statistics can mix different building types and sensor regimes.

Traffic retains Day 1 zone metadata. Optional zone metadata nulls are reported separately from required-field failures. Raw measurements have not been imputed or clipped by Day 1.

In [10]:
def compact_report(report):
    return {k: v for k, v in report.items() if k != "numeric_summary"}

def profile_data(df, kind):
    report = validate_dataframe(df, kind)
    numeric = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
    text = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    extras = {}
    for c in numeric:
        extras[c + "_nonfinite"] = df.filter(
            F.isnan(F.col(c)) | (F.abs(F.col(c)) == float("inf"))).count()
    for c in text:
        extras[c + "_blank"] = df.filter(F.trim(F.col(c)) == "").count()
    report["additional_issues"] = {k: v for k, v in extras.items() if v}
    report["categorical_cardinality"] = {
        c: df.select(c).distinct().count() for c in text if c != "timestamp"}
    if "timestamp" in df.columns:
        key = get_validation_config(kind)["duplicate_columns"][0]
        times = df.select(key, F.try_to_timestamp("timestamp").alias("t")).dropna()
        times = times.dropDuplicates([key, "t"])
        w = Window.partitionBy(key).orderBy("t")
        intervals = times.withColumn("seconds", F.col("t").cast("long") -
                                     F.lag("t").over(w).cast("long"))
        report["observed_intervals_seconds"] = intervals.select("seconds").summary().toPandas().to_dict("records")
    return report

input_frames = {}
before = {}
for kind, path in INPUT_PATHS.items():
    df = spark.read.parquet(str(path)).cache()
    if df.count() != source_records[kind]["rows"]:
        raise ValueError(f"{kind}: row count differs from the Day 1 manifest.")
    input_frames[kind] = df
    before[kind] = profile_data(df, kind)
    print(kind, compact_report(before[kind]))
display(pd.DataFrame([{"dataset": k, "rows": r["record_count"],
                       "contract_valid": r["valid"],
                       "duplicate_rows": r["duplicate_count"],
                       "null_cells": sum(r["null_counts"].values()),
                       "additional_issues": r["additional_issues"]}
                      for k, r in before.items()]))

city_zones {'dataset_type': 'city_zones', 'valid': True, 'record_count': 36000, 'missing_columns': [], 'non_numeric_columns': [], 'null_counts': {}, 'duplicate_count': 0, 'range_violations': {}, 'value_violations': {}, 'timestamp_violations': {}, 'cross_field_violations': {}, 'additional_issues': {}, 'categorical_cardinality': {'zone_id': 36000, 'zone_name': 36000, 'zone_type': 6}}
traffic {'dataset_type': 'traffic', 'valid': True, 'record_count': 36000, 'missing_columns': [], 'non_numeric_columns': [], 'null_counts': {}, 'duplicate_count': 0, 'range_violations': {}, 'value_violations': {}, 'timestamp_violations': {}, 'cross_field_violations': {}, 'additional_issues': {}, 'categorical_cardinality': {'sensor_id': 3000, 'congestion_level': 3, 'road_type': 5, 'zone_id': 35962, 'zone_name': 35962, 'zone_type': 6}, 'observed_intervals_seconds': [{'summary': 'count', 'seconds': '33000'}, {'summary': 'mean', 'seconds': '900000.0'}, {'summary': 'stddev', 'seconds': '0.0'}, {'summary': 'min', '

,dataset,rows,contract_valid,duplicate_rows,null_cells,additional_issues
0,city_zones,36000,True,0,0,{}
1,traffic,36000,True,0,0,{}
2,air_quality,36000,True,0,0,{}
3,weather,36000,True,0,0,{}
4,energy,36000,True,0,0,{}
5,occupancy,36000,True,0,0,{}
6,fiscal,36000,True,0,0,{}


## 2a. Instructor alignment: sensor diagnostics and missing-data patterns

The instructor's outline asks for sensor-level diagnostics as well as dataset-wide
profiles. The following reports measure missing/non-finite values per sensor,
hour and weekday; observed reporting intervals; constant readings; and freshness
relative to the dataset's latest timestamp (not today's date for historical data).

The **completeness score** is the percentage of present, finite measurement cells.
It is not a calibrated sensor-health probability. Status is a review triage:
missing measurements or constant readings warrant review, not a malfunction diagnosis.
Constant-series tests require at least three observed values. Near-constant/noisy
thresholds require sensor-specific evidence and are not guessed.

Expected reporting intervals default to unknown. Only a confirmed per-sensor
schedule enables a long-interval flag; observed cadence alone does not establish outages.
Rows absent altogether cannot appear in null-cell percentages.


In [11]:
# A confirmed reporting schedule must refer to EACH sensor, not the generator's dataset step.
EXPECTED_INTERVAL_SECONDS = {kind: None for kind in INPUT_PATHS}
DIAGNOSTIC_MEASUREMENTS = {
    "traffic": ["vehicle_count", "avg_speed"],
    "air_quality": ["pm25", "pm10", "no2", "co", "temperature", "humidity"],
    "weather": ["temperature", "humidity", "wind_speed", "precipitation", "pressure"],
    "energy": ["power_consumption", "voltage", "current", "power_factor"],
    "occupancy": ["available_rooms", "occupied_rooms", "guests"],
    "fiscal": ["expense", "revenue"],
}
def missing_measurement(c):
    return F.col(c).isNull() | F.isnan(F.col(c)) | (F.abs(F.col(c)) == float("inf"))

def sensor_diagnostics(df, sensor_col, columns, expected_seconds=None):
    if expected_seconds is not None and (not math.isfinite(expected_seconds) or expected_seconds <= 0):
        raise ValueError("Expected interval must be finite and positive")
    expressions = [F.count("*").alias("readings"),
                   F.min("timestamp").alias("first_reading"),
                   F.max("timestamp").alias("last_reading")]
    for c in columns:
        valid = F.when(~missing_measurement(c), F.col(c))
        expressions += [F.sum(missing_measurement(c).cast("long")).alias(c + "_missing"),
                        F.count(valid).alias(c + "_observed"),
                        F.variance(valid).alias(c + "_variance")]
    health = df.groupBy(sensor_col).agg(*expressions)
    missing_total = F.lit(0)
    constant = F.lit(False)
    for c in columns:
        health = health.withColumn(c + "_missing_pct", 100 * F.col(c + "_missing") / F.col("readings"))
        missing_total = missing_total + F.col(c + "_missing")
        constant = constant | ((F.col(c + "_observed") >= 3) & (F.col(c + "_variance") == 0))
    health = (health.withColumn("completeness_score", 100 * (1 - missing_total / (F.col("readings") * len(columns))))
              .withColumn("constant_measurement_flag", F.coalesce(constant, F.lit(False))))
    times = df.select(sensor_col, "timestamp").dropna().dropDuplicates()
    w = Window.partitionBy(sensor_col).orderBy("timestamp")
    gaps = times.withColumn("interval_seconds", F.col("timestamp").cast("long") -
                            F.lag("timestamp").over(w).cast("long"))
    interval_stats = gaps.groupBy(sensor_col).agg(
        F.count("interval_seconds").alias("observed_intervals"),
        F.percentile_approx("interval_seconds", 0.5).alias("median_interval_seconds"),
        F.max("interval_seconds").alias("max_interval_seconds"))
    latest = df.agg(F.max(F.col("timestamp").cast("long"))).first()[0]
    health = (health.join(interval_stats, sensor_col, "left")
              .withColumn("hours_behind_dataset_latest",
                          (F.lit(latest) - F.col("last_reading").cast("long")) / 3600)
              .withColumn("schedule_assessed", F.lit(expected_seconds is not None))
              .withColumn("long_interval_flag",
                          (F.col("max_interval_seconds") > 1.5 * expected_seconds)
                          if expected_seconds is not None else F.lit(None).cast("boolean")))
    review = ((F.col("completeness_score") < 100) | F.col("constant_measurement_flag") |
              F.coalesce(F.col("long_interval_flag"), F.lit(False)))
    return health.withColumn("review_status", F.when(review, "review").otherwise("no_flag_in_assessed_checks"))

def missing_patterns(df, group_col, columns):
    enriched = df.withColumn("hour", F.hour("timestamp")).withColumn("weekday", F.dayofweek("timestamp"))
    parts = []
    for c in columns:
        part = enriched.groupBy(group_col).agg(
            F.count("*").alias("rows"),
            F.sum(missing_measurement(c).cast("long")).alias("missing_cells"))
        parts.append(part.withColumn("measurement", F.lit(c))
                     .withColumn("missing_pct", 100 * F.col("missing_cells") / F.col("rows")))
    result = parts[0]
    for part in parts[1:]:
        result = result.unionByName(part)
    return result

sensor_health, missing_pattern_reports, categorical_top_values = {}, {}, {}
for kind, frame in input_frames.items():
    text_cols = [f.name for f in frame.schema.fields if isinstance(f.dataType, StringType)]
    categorical_top_values[kind] = {
        c: frame.groupBy(c).count().orderBy(F.desc("count"), F.col(c)).limit(5).toPandas().to_dict("records")
        for c in text_cols if before[kind]["categorical_cardinality"].get(c, 999) < 20
    }
    if kind not in DIAGNOSTIC_MEASUREMENTS:
        continue
    key = get_validation_config(kind)["duplicate_columns"][0]
    cols = DIAGNOSTIC_MEASUREMENTS[kind]
    sensor_health[kind] = sensor_diagnostics(frame, key, cols, EXPECTED_INTERVAL_SECONDS[kind])
    missing_pattern_reports[kind] = {group: missing_patterns(frame, group, cols)
                                    for group in ("hour", "weekday")}
    # Per-sensor missing percentages are already included in sensor_health.
    print(kind, "sensor diagnostic status:")
    sensor_health[kind].groupBy("review_status").count().show(truncate=False)


traffic sensor diagnostic status:
+--------------------------+-----+
|review_status             |count|
+--------------------------+-----+
|no_flag_in_assessed_checks|3000 |
+--------------------------+-----+

air_quality sensor diagnostic status:
+--------------------------+-----+
|review_status             |count|
+--------------------------+-----+
|no_flag_in_assessed_checks|1800 |
+--------------------------+-----+

weather sensor diagnostic status:
+--------------------------+-----+
|review_status             |count|
+--------------------------+-----+
|no_flag_in_assessed_checks|1200 |
+--------------------------+-----+

energy sensor diagnostic status:
+--------------------------+-----+
|review_status             |count|
+--------------------------+-----+
|no_flag_in_assessed_checks|2400 |
+--------------------------+-----+

occupancy sensor diagnostic status:
+--------------------------+-----+
|review_status             |count|
+--------------------------+-----+
|no_flag_in_asse

## 3. Cleaning and missing data policy
Normalize whitespace and configured categories, parse timestamps, and convert non-finite numeric values to null. Required nulls, impossible ranges, invalid categories and cross-field failures are quarantined with reasons. Exact duplicate rows are collapsed; all remaining rows sharing a primary key are quarantined because picking a winner would require provenance.

No blanket mean filling or zero substitution is used: a missing fiscal value is not zero revenue. The optional interpolation function fills existing null cells only between two observed values for the **same sensor** when the entire bracket is at most one hour. It does not create missing timestamps or extrapolate across long outages. Apply only to approved continuous measurements before cleaning; it is demonstrated on a fixture below, not automatically applied to production data.

In [12]:
def interpolate_short_gaps(df, value_col, sensor_col, max_gap_seconds=3600):
    if max_gap_seconds <= 0:
        raise ValueError("max_gap_seconds must be positive")
    w = Window.partitionBy(sensor_col).orderBy("timestamp")
    known = F.when(F.col(value_col).isNotNull(), F.struct(
        F.col("timestamp").cast("long").alias("t"), F.col(value_col).cast("double").alias("v")))
    result = (df.withColumn("_prev", F.last(known, ignorenulls=True).over(
                  w.rowsBetween(Window.unboundedPreceding, -1)))
              .withColumn("_next", F.first(known, ignorenulls=True).over(
                  w.rowsBetween(1, Window.unboundedFollowing))))
    gap = F.col("_next.t") - F.col("_prev.t")
    fill = (F.col(value_col).isNull() & (gap > 0) & (gap <= max_gap_seconds))
    interpolated = F.col("_prev.v") + (
        (F.col("timestamp").cast("long") - F.col("_prev.t")) / gap
    ) * (F.col("_next.v") - F.col("_prev.v"))
    return (result.withColumn(value_col + "_imputed", F.coalesce(fill, F.lit(False)))
            .withColumn(value_col, F.when(fill, interpolated).otherwise(F.col(value_col)))
            .drop("_prev", "_next"))

def clean_data(df, kind):
    config = get_validation_config(kind)
    required = config["required_columns"]
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"{kind}: missing columns {sorted(missing)}; schema repair required")
    # Preserve a row-level fingerprint before normalization.
    result = df.withColumn("_raw_hash", F.sha2(F.to_json(F.struct(
        *[F.col(c) for c in sorted(df.columns)])), 256))
    for field in df.schema.fields:
        c = field.name
        if isinstance(field.dataType, StringType):
            result = result.withColumn(c, F.when(F.trim(F.col(c)) != "", F.trim(F.col(c))))
        if isinstance(field.dataType, NumericType):
            result = result.withColumn(c, F.when(
                ~(F.isnan(F.col(c)) | (F.abs(F.col(c)) == float("inf"))), F.col(c)))
    for c in config.get("allowed_values", {}):
        result = result.withColumn(c, F.lower(F.col(c)))
    if "timestamp" in required:
        result = result.withColumn("timestamp", F.try_to_timestamp("timestamp"))
    types = {f.name: f.dataType for f in result.schema.fields}
    wrong_types = [c for c in config.get("ranges", {}) if not isinstance(types[c], NumericType)]
    if wrong_types:
        raise ValueError(f"{kind}: review nonnumeric columns {wrong_types}; no silent coercion")
    reasons = [F.when(F.col(c).isNull(), F.lit("missing:" + c)) for c in required]
    for c, limits in config.get("ranges", {}).items():
        invalid = F.lit(False)
        if "min" in limits:
            invalid = invalid | (F.col(c) < limits["min"])
        if "max" in limits:
            invalid = invalid | (F.col(c) > limits["max"])
        reasons.append(F.when(invalid, F.lit("range:" + c)))
    for c, allowed in config.get("allowed_values", {}).items():
        reasons.append(F.when(~F.col(c).isin(allowed), F.lit("category:" + c)))
    for rule in config.get("cross_field_rules", []):
        reasons.append(F.when(F.col(rule["left"]) > F.col(rule["right"]), F.lit(rule["name"])))
    result = result.withColumn("_reasons", F.concat_ws(";", *reasons))
    invalid_rows = result.filter(F.col("_reasons") != "")
    eligible = result.filter(F.col("_reasons") == "")
    # Deterministic survivor for identical normalized values.
    w = Window.partitionBy(*df.columns).orderBy("_raw_hash")
    ranked = eligible.withColumn("_duplicate_rank", F.row_number().over(w))
    duplicates = (ranked.filter(F.col("_duplicate_rank") > 1).drop("_duplicate_rank")
                  .withColumn("_reasons", F.lit("exact_duplicate")))
    distinct = ranked.filter(F.col("_duplicate_rank") == 1).drop("_duplicate_rank")
    key_window = Window.partitionBy(*config["duplicate_columns"])
    distinct = distinct.withColumn("_key_count", F.count("*").over(key_window))
    conflicts = (distinct.filter(F.col("_key_count") > 1).drop("_key_count")
                 .withColumn("_reasons", F.lit("conflicting_primary_key")))
    clean = distinct.filter(F.col("_key_count") == 1).drop("_key_count", "_reasons")
    quarantine = invalid_rows.unionByName(duplicates).unionByName(conflicts)
    return clean, quarantine

## 4. Outlier detection and treatment
Fit IQR (1.5 × interquartile range) and absolute Z-score (>3) thresholds on finite, eligible measurements. Zero IQR or zero standard deviation yields no flags for that method; domain rules still apply. Coordinates, keys and counts are excluded.

Default treatment is `flag`. Optional `clip` caps approved measurements at IQR bounds, preserving originals and per-column treatment flags. These are global exploratory thresholds, not calibrated anomaly models: review by sensor/building type before enabling clipping. Fiscal values are flagged but never clipped by the default configuration.

In [13]:
MEASUREMENTS = {
    "traffic": ["avg_speed"], "air_quality": ["pm25", "pm10", "no2", "co", "temperature", "humidity"],
    "weather": ["temperature", "humidity", "wind_speed", "precipitation", "pressure"],
    "energy": ["power_consumption", "voltage", "current", "power_factor"],
    "city_zones": [], "occupancy": [], "fiscal": ["expense", "revenue"],
}

def fit_outlier_bounds(df, columns):
    bounds = {}
    for c in columns:
        q = df.approxQuantile(c, [0.25, 0.75], 0.001)
        if not q:
            continue
        stats = df.agg(F.avg(c).alias("mean"), F.stddev_pop(c).alias("std")).first()
        iqr = q[1] - q[0]
        bounds[c] = {"low": q[0] - 1.5 * iqr, "high": q[1] + 1.5 * iqr,
                     "iqr": iqr, "mean": stats["mean"], "std": stats["std"]}
    return bounds

def treat_outliers(df, bounds, mode="flag", clip_columns=()):
    if mode not in {"flag", "clip"}:
        raise ValueError("mode must be flag or clip")
    result = df
    for c, b in bounds.items():
        x = F.col(c)
        iqr_flag = ((x < b["low"]) | (x > b["high"])) if b["iqr"] > 0 else F.lit(False)
        z_flag = (F.abs((x - b["mean"]) / b["std"]) > 3) if b["std"] else F.lit(False)
        result = (result.withColumn(c + "_iqr_outlier", F.coalesce(iqr_flag, F.lit(False)))
                  .withColumn(c + "_zscore_outlier", F.coalesce(z_flag, F.lit(False))))
        if mode == "clip" and c in clip_columns:
            result = (result.withColumn(c + "_original", x)
                      .withColumn(c + "_treated", F.col(c + "_iqr_outlier"))
                      .withColumn(c, F.when(F.col(c + "_iqr_outlier"),
                          F.greatest(F.lit(b["low"]), F.least(x, F.lit(b["high"])))).otherwise(x)))
    return result

def standardize_units(df, units):
    # Add explicitly named analysis columns; preserve source contract columns.
    conversions = {
        "temperature": {"F": ("temperature_celsius", lambda x: (x - 32) * 5 / 9),
                        "C": ("temperature_celsius", lambda x: x)},
        "avg_speed": {"mph": ("speed_kmh", lambda x: x * 1.609344),
                      "km/h": ("speed_kmh", lambda x: x)},
        "power_consumption": {"W": ("power_kw", lambda x: x / 1000),
                              "kW": ("power_kw", lambda x: x)},
    }
    for c, unit in units.items():
        if c not in df.columns or unit is None:
            continue
        if unit not in conversions[c]:
            raise ValueError(f"Unsupported unit for {c}: {unit}")
        name, conversion = conversions[c][unit]
        df = df.withColumn(name, conversion(F.col(c)))
    return df

## 4a. Compare optional treatment strategies

The instructor demonstrates flagging, capping, removal and median replacement.
Production still uses flag-only by default so Day 3/4 retain genuine extremes.
The helper below demonstrates removal and median replacement on an explicit
measurement/flag pair; median replacement uses only unflagged finite values.
These alternatives are not applied to the seven production datasets.

Our clipping uses IQR bounds rather than the outline's 5th/95th percentiles.
Our Z-scores use population standard deviation rather than sample standard
deviation. Those are explicit method choices, not identical implementations.
Domain validation remains in the shared contract; the outline's example upper
limits for speed, PM2.5 and temperature require confirmed units and local review.


In [14]:
def alternative_outlier_treatment(df, column, flag_col, strategy):
    if strategy == "remove":
        return df.filter(~F.coalesce(F.col(flag_col), F.lit(False)))
    if strategy != "impute":
        raise ValueError("Use remove or impute")
    eligible = df.filter(~F.coalesce(F.col(flag_col), F.lit(False)) & ~missing_measurement(column))
    quantiles = eligible.approxQuantile(column, [0.5], 0.0)
    if not quantiles:
        raise ValueError("No unflagged finite values for median replacement")
    return (df.withColumn(column + "_original", F.col(column))
            .withColumn(column + "_imputed", F.coalesce(F.col(flag_col), F.lit(False)))
            .withColumn(column, F.when(F.col(column + "_imputed"), F.lit(quantiles[0])).otherwise(F.col(column))))


## 5. Check behavior with deliberately dirty examples
The generated files may have no contract violations. These small fixtures exercise cleaning, conflicting keys, missing data interpolation and treatment so a clean input does not hide an inactive pipeline.

In [15]:
fixture = spark.createDataFrame([
    (" a ", "2025-01-01 00:00:00", 40.0, -73.0, 10, 25.0, " LOW ", "road"),
    (" a ", "2025-01-01 00:00:00", 40.0, -73.0, 10, 25.0, " LOW ", "road"),
    ("b", "bad", 40.0, -73.0, 10, 25.0, "low", "road"),
    ("c", "2025-01-01 00:00:00", 95.0, -73.0, 10, 25.0, "low", "road"),
    ("d", "2025-01-01 00:00:00", 40.0, -73.0, 10, float("inf"), "low", "road"),
    ("e", "2025-01-01 00:00:00", 40.0, -73.0, 10, 25.0, "low", "road"),
    ("e", "2025-01-01 00:00:00", 40.0, -73.0, 10, 30.0, "low", "road"),
], "sensor_id string, timestamp string, location_lat double, location_lon double, vehicle_count int, avg_speed double, congestion_level string, road_type string")
good, rejected = clean_data(fixture, "traffic")
assert good.count() == 1 and rejected.count() == 6
assert validate_dataframe(good, "traffic")["valid"]
assert good.first()["sensor_id"] == "a"
outlier_fixture = spark.createDataFrame([(float(x),) for x in range(1, 21)] + [(1000.0,)], ["reading"])
bounds = fit_outlier_bounds(outlier_fixture, ["reading"])
flagged = treat_outliers(outlier_fixture, bounds)
assert flagged.filter("reading_iqr_outlier").count() == 1
assert flagged.agg(F.max("reading")).first()[0] == 1000
clipped = treat_outliers(outlier_fixture, bounds, "clip", ["reading"])
assert clipped.agg(F.max("reading")).first()[0] < 1000
assert clipped.filter("reading_treated").count() == 1
constant = spark.createDataFrame([(5.0,), (5.0,)], ["reading"])
assert treat_outliers(constant, fit_outlier_bounds(constant, ["reading"])).filter("reading_iqr_outlier OR reading_zscore_outlier").count() == 0
gap_fixture = spark.createDataFrame([
    ("a", "2025-01-01 00:00:00", 0.0), ("a", "2025-01-01 00:10:00", None),
    ("a", "2025-01-01 00:20:00", 20.0), ("a", "2025-01-01 03:00:00", None),
    ("a", "2025-01-01 06:00:00", 60.0), ("b", "2025-01-01 00:10:00", None)],
    "sensor_id string, timestamp string, reading double").withColumn("timestamp", F.to_timestamp("timestamp"))
filled = interpolate_short_gaps(gap_fixture, "reading", "sensor_id")
assert filled.filter("reading_imputed").first()["reading"] == 10.0
assert filled.filter("reading IS NULL").count() == 2
assert standardize_units(spark.createDataFrame([(32.0,)], ["temperature"]),
                         {"temperature": "F"}).first()["temperature_celsius"] == 0.0
print("Cleaning, deduplication, outlier, interpolation and unit-conversion checks passed.")
assert alternative_outlier_treatment(flagged, "reading", "reading_iqr_outlier", "remove").count() == 20
imputed_outlier = alternative_outlier_treatment(flagged, "reading", "reading_iqr_outlier", "impute")
assert imputed_outlier.count() == 21
assert imputed_outlier.filter("reading_imputed").count() == 1
assert imputed_outlier.filter("reading_imputed").first()["reading"] == 10.0
diagnostic_fixture = spark.createDataFrame([
    ("a", "2025-01-01 00:00:00", 5.0), ("a", "2025-01-01 00:10:00", 5.0),
    ("a", "2025-01-01 00:20:00", 5.0), ("a", "2025-01-01 02:00:00", None),
    ("b", "2025-01-01 00:00:00", float("inf")),
], "sensor_id string, timestamp string, reading double").withColumn("timestamp", F.to_timestamp("timestamp"))
diagnostics = sensor_diagnostics(diagnostic_fixture, "sensor_id", ["reading"], 600)
a_health = diagnostics.filter("sensor_id = 'a'").first()
assert a_health["completeness_score"] == 75.0
assert a_health["constant_measurement_flag"] and a_health["long_interval_flag"]
assert diagnostics.filter("sensor_id = 'b'").first()["completeness_score"] == 0.0
assert sensor_diagnostics(diagnostic_fixture, "sensor_id", ["reading"]).filter("long_interval_flag IS NOT NULL").count() == 0
assert missing_patterns(diagnostic_fixture, "hour", ["reading"]).agg(F.sum("missing_cells")).first()[0] == 2
print("Sensor diagnostics, missing patterns and alternative-treatment checks passed.")


Cleaning, deduplication, outlier, interpolation and unit-conversion checks passed.
Sensor diagnostics, missing patterns and alternative-treatment checks passed.


## 6. Run the pipeline and publish local analysis datasets
Every source row is accounted for as accepted or quarantined (including removed duplicate copies). Lineage includes the source file, input fingerprint, run ID, policy and processing time. Accepted output must pass the shared validator and extra finite-value checks before publication. Each Parquet path is a Spark directory containing part files.

Unknown units remain explicitly recorded as unknown in the manifest. Analysis involving unit comparisons or cross-sensor joins still requires agreement on units, timezone and sensor geography.

In [16]:
OUTPUT.mkdir(parents=True, exist_ok=False)
after, audit, thresholds = {}, [], {}
clean_frames = {}
for kind, source in input_frames.items():
    cleaned, quarantined = clean_data(source, kind)
    cleaned, quarantined = cleaned.cache(), quarantined.cache()
    accepted, rejected = cleaned.count(), quarantined.count()
    assert accepted + rejected == before[kind]["record_count"], kind
    thresholds[kind] = fit_outlier_bounds(cleaned, MEASUREMENTS[kind])
    analysis = treat_outliers(cleaned, thresholds[kind], OUTLIER_TREATMENT, CLIP_COLUMNS.get(kind, []))
    analysis = standardize_units(analysis, SOURCE_UNITS)
    analysis = (analysis.withColumn("_source_file", F.lit(str(INPUT_PATHS[kind].relative_to(ROOT))))
                .withColumn("_day1_run_id", F.lit(DAY1.name))
                .withColumn("_run_id", F.lit(RUN_ID))
                .withColumn("_outlier_policy", F.lit(OUTLIER_TREATMENT))
                .withColumn("_processed_at", F.lit(RUN_ID)))
    after[kind] = profile_data(analysis, kind)
    if not after[kind]["valid"] or after[kind]["additional_issues"]:
        raise ValueError(f"{kind}: output failed validation; inspect report before publishing")
    flag_cols = [c for c in analysis.columns if c.endswith(("_iqr_outlier", "_zscore_outlier"))]
    flag_counts = analysis.agg(*[F.sum(F.col(c).cast("long")).alias(c) for c in flag_cols]).first().asDict() if flag_cols else {}
    quarantine_counts = {r["_reasons"]: r["count"] for r in quarantined.groupBy("_reasons").count().collect()}
    analysis.write.mode("errorifexists").parquet(str(OUTPUT / kind))
    quarantined.write.mode("errorifexists").parquet(str(OUTPUT / "quarantine" / kind))
    # Verify persisted files, not only in-memory frames.
    persisted = spark.read.parquet(str(OUTPUT / kind))
    assert persisted.count() == accepted
    assert validate_dataframe(persisted, kind)["valid"]
    if kind == "traffic":
        assert {"zone_id", "zone_name", "zone_type", "zone_match_count"} <= set(persisted.columns)
        zone_cols = ["sensor_id", "timestamp", "zone_id", "zone_name", "zone_type", "zone_match_count"]
        assert analysis.select(*zone_cols).exceptAll(persisted.select(*zone_cols)).limit(1).count() == 0
    clean_frames[kind] = persisted
    audit.append({"dataset": kind, "source_rows": before[kind]["record_count"],
                  "accepted_rows": accepted, "quarantined_rows": rejected,
                  "valid_after": after[kind]["valid"],
                  "outlier_flags": flag_counts, "quarantine_reasons": quarantine_counts})
    cleaned.unpersist()
    quarantined.unpersist()
    source.unpersist()
    print(kind, "accepted:", accepted, "quarantined:", rejected)

summary = pd.DataFrame(audit)
display(summary[["dataset", "source_rows", "accepted_rows", "quarantined_rows", "valid_after"]])

city_zones accepted: 36000 quarantined: 0


traffic accepted: 36000 quarantined: 0
air_quality accepted: 36000 quarantined: 0
weather accepted: 36000 quarantined: 0
energy accepted: 36000 quarantined: 0
occupancy accepted: 36000 quarantined: 0
fiscal accepted: 36000 quarantined: 0


,dataset,source_rows,accepted_rows,quarantined_rows,valid_after
0,city_zones,36000,36000,0,True
1,traffic,36000,36000,0,True
2,air_quality,36000,36000,0,True
3,weather,36000,36000,0,True
4,energy,36000,36000,0,True
5,occupancy,36000,36000,0,True
6,fiscal,36000,36000,0,True


## 7. Save the assessment report and run manifest
The JSON contains full numerical summaries, per-column failures, observed intervals, outlier thresholds and row accounting. The Markdown report is the short team handoff. The manifest hashes the local input files so counts alone are not mistaken for content equality. A success manifest is written only after every dataset has completed; interrupted run directories are incomplete.

In [17]:
# Persist diagnostics without collecting every sensor row into notebook output.
diagnostic_counts = {}
for kind, health in sensor_health.items():
    health.write.mode("errorifexists").parquet(str(OUTPUT / "diagnostics" / kind / "sensor_health"))
    diagnostic_counts[kind] = health.count()
    for group, report in missing_pattern_reports[kind].items():
        report.write.mode("errorifexists").parquet(str(OUTPUT / "diagnostics" / kind / ("missing_by_" + group)))
completion_checklist = {
    "all_seven_datasets_profiled": len(before) == 7,
    "six_sensor_datasets_assessed": len(sensor_health) == 6,
    "missing_patterns_reported": len(missing_pattern_reports) == 6,
    "interpolation_fixture_passed": True,
    "outlier_treatment_fixtures_passed": True,
    "all_outputs_validated": all(r["valid_after"] for r in audit),
    "quality_scores_calculated": len(diagnostic_counts) == 6,
    "unit_conversion_test_passed": True,
    "physical_units_confirmed": all(v is not None for v in SOURCE_UNITS.values()),
    "reporting_schedules_confirmed": all(EXPECTED_INTERVAL_SECONDS[k] is not None for k in DIAGNOSTIC_MEASUREMENTS),
}
# False evidence-confirmation entries are open questions, not failed execution tests.
(OUTPUT / "completion_checklist.json").write_text(json.dumps(completion_checklist, indent=2))
assessment = {"before": before, "after": after, "audit": audit, "thresholds": thresholds,
              "categorical_top_values": categorical_top_values, "sensor_diagnostic_counts": diagnostic_counts,
              "completion_checklist": completion_checklist}
(OUTPUT / "quality_assessment.json").write_text(json.dumps(assessment, indent=2, default=str))
lines = ["# Day 2 quality assessment", "", f"Run: {RUN_ID}", f"Day 1 source: {DAY1.name}", "",
         "| Dataset | Day 1 input | Accepted | Quarantined | Valid after |",
         "|---|---:|---:|---:|---|"]
for row in audit:
    lines.append("| {dataset} | {source_rows} | {accepted_rows} | {quarantined_rows} | {valid_after} |".format(**row))
lines += ["", f"Outlier treatment: {OUTLIER_TREATMENT}. IQR and Z-score counts may overlap.",
          "See quality_assessment.json for thresholds, flags, numeric profiles and rejection reasons.",
          "", "Units are unconfirmed unless explicitly configured. Timestamp timezone is unconfirmed.",
          "Observed sensor intervals do not establish outages. No S2 reconciliation was performed.",
          "Review quarantined records before any backfill. Statistical extremes may be valid events."]
lines += ["", "## Instructor-aligned diagnostic additions",
          "Per-sensor completeness, variance, relative freshness and observed cadence: diagnostics/<dataset>/sensor_health.",
          "Missing measurement percentages by hour and weekday: diagnostics/<dataset>/missing_by_*.",
          "Status is review triage, not proof of sensor malfunction. Unknown schedules do not establish outages.",
          "Completion and unresolved evidence: completion_checklist.json.",
          "Treatment removal/median replacement and bounded interpolation passed dirty-fixture tests; production remains flag-only by default."]
(OUTPUT / "quality_assessment.md").write_text("\n".join(lines) + "\n")
assert all(input_hash(path) == source_records[k]["sha256"] for k, path in INPUT_PATHS.items()), "Day 1 inputs changed during this run."
assert hashlib.sha256((DAY1 / "manifest.json").read_bytes()).hexdigest() == source_manifest_hash
manifest = {"status": "complete", "run_id": RUN_ID, "python": sys.version,
            "spark": spark.version, "source_units": SOURCE_UNITS,
            "day1_run": DAY1.name, "day1_manifest_sha256": source_manifest_hash,
            "timestamp_semantics": "source wall-clock; source timezone unconfirmed",
            "outlier_policy": OUTLIER_TREATMENT,
            "source_sha256": {k: source_records[k]["sha256"] for k in INPUT_PATHS},
            "datasets": audit}
(OUTPUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print((OUTPUT / "quality_assessment.md").read_text())
print("Completed output:", OUTPUT)

# Day 2 quality assessment

Run: 20260914T135902169587Z
Day 1 source: 20260914T135810735977Z

| Dataset | Day 1 input | Accepted | Quarantined | Valid after |
|---|---:|---:|---:|---|
| city_zones | 36000 | 36000 | 0 | True |
| traffic | 36000 | 36000 | 0 | True |
| air_quality | 36000 | 36000 | 0 | True |
| weather | 36000 | 36000 | 0 | True |
| energy | 36000 | 36000 | 0 | True |
| occupancy | 36000 | 36000 | 0 | True |
| fiscal | 36000 | 36000 | 0 | True |

Outlier treatment: flag. IQR and Z-score counts may overlap.
See quality_assessment.json for thresholds, flags, numeric profiles and rejection reasons.

Units are unconfirmed unless explicitly configured. Timestamp timezone is unconfirmed.
Observed sensor intervals do not establish outages. No S2 reconciliation was performed.
Review quarantined records before any backfill. Statistical extremes may be valid events.

## Instructor-aligned diagnostic additions
Per-sensor completeness, variance, relative freshness and observed cadenc

## 8. Day 3 handoff
Load an accepted dataset with `spark.read.parquet(str(OUTPUT / "traffic"))`. The original measurement columns retain their source meaning; additional boolean flags support sensitivity analysis with and without extremes. Use the manifest to identify the input snapshot and approved unit conversions.

For a later session, set `OUTPUT` to the completed run directory printed above and confirm `manifest.json` says `complete`. Keep large generated outputs local; only commit the notebook unless the team has agreed on artifact storage. This notebook performs no database writes.

Rerun Day 3 to use this new Day 2 run, and rerun Day 4 after Day 3 completes. Existing downstream runs remain valid historical snapshots; they do not update automatically. Their manifests identify which versions they used.

In [18]:
clean_frames["traffic"].select("sensor_id", "timestamp", "avg_speed",
                               "avg_speed_iqr_outlier", "avg_speed_zscore_outlier").show(5)
# Run spark.stop() when you have finished using Spark in this kernel.

+---------+-------------------+---------+---------------------+------------------------+
|sensor_id|          timestamp|avg_speed|avg_speed_iqr_outlier|avg_speed_zscore_outlier|
+---------+-------------------+---------+---------------------+------------------------+
| TRF-0001|2025-04-15 04:00:00|    21.26|                false|                   false|
| TRF-0002|2025-01-11 10:05:00|    31.99|                false|                   false|
| TRF-0002|2025-03-04 12:05:00|    15.08|                false|                   false|
| TRF-0002|2025-03-14 22:05:00|    39.66|                false|                   false|
| TRF-0002|2025-03-25 08:05:00|    24.43|                false|                   false|
+---------+-------------------+---------+---------------------+------------------------+
only showing top 5 rows
